# LeetCode #1149 — Article Views II (Lượt xem bài viết II)

**Medium · Database / SQL · Premium 🔒 · GROUP BY · COUNT(DISTINCT) · HAVING**  
[Trang đề LeetCode](https://leetcode.com/problems/article-views-ii/) · [Đối chiếu đề và ví dụ](https://leetcode.doocs.org/en/lc/1149/)

**Phương pháp học:** học SQL từ căn bản, chạy từng truy vấn trên SQLite bằng Python (`sqlite3`), cuối bài cung cấp câu **MySQL** đúng để nộp LeetCode.

**Câu hỏi xuyên suốt:** Làm sao tìm người xem **hơn một bài viết KHÁC NHAU trong CÙNG MỘT NGÀY**, kể cả khi bảng bị lặp hàng?

## 1. Mục tiêu

1. Hiểu bảng dữ liệu, cột, hàng, `SELECT`, `WHERE`, `ORDER BY`.
2. Nhận ra **hai lớp trùng lặp**: trùng lượt xem bài và một người đạt điều kiện ở nhiều ngày.
3. Biết `GROUP BY viewer_id, view_date` tạo từng nhóm **người–ngày**.
4. Phân biệt `COUNT(*)`, `COUNT(article_id)`, `COUNT(DISTINCT article_id)`.
5. Hiểu `HAVING` lọc nhóm, khác `WHERE` lọc từng hàng.
6. Viết câu SQL đúng cho LeetCode, tự kiểm thử dữ liệu khác và hoàn thành 5 bài tập.

## 2. Đề bài và lược đồ bảng `Views`

Bảng có 4 cột:

| Cột | Kiểu | Ý nghĩa |
|---|---|---|
| `article_id` | `int` | Mã bài viết |
| `author_id` | `int` | Mã người viết |
| `viewer_id` | `int` | Mã người xem |
| `view_date` | `date` | Ngày người đó xem |

**Bảng có thể có các hàng trùng nhau.** Người viết có thể tự xem bài của mình (`author_id == viewer_id`), đây **không phải** điều kiện lọc của bài II.

**Yêu cầu:** Liệt kê những `viewer_id` đã xem **ít nhất 2 bài khác nhau trong cùng một ngày**, kết quả gồm **một cột tên `id`**, **không có người trùng**, **sắp xếp tăng dần `id`**.

**Lưu ý nguồn:** #1149 là bài **Premium**. Notebook đối chiếu mô tả/ ví dụ công khai trên LeetCode Wiki. Không tự thêm ràng buộc ngoài đề.

## 3. Ví dụ gốc (8 hàng) — đọc bằng tay

| article_id | author_id | viewer_id | view_date |
|---:|---:|---:|---|
| 1 | 3 | 5 | 2019-08-01 |
| 3 | 4 | 5 | 2019-08-01 |
| 1 | 3 | 6 | 2019-08-02 |
| 2 | 7 | 7 | 2019-08-01 |
| 2 | 7 | 6 | 2019-08-02 |
| 4 | 7 | 1 | 2019-07-22 |
| 3 | 4 | 4 | 2019-07-21 |
| 3 | 4 | 4 | 2019-07-21 |

- Người `5`: bài **1 và 3** trong ngày `2019-08-01` ⇒ đạt.
- Người `6`: bài **1 và 2** trong ngày `2019-08-02` ⇒ đạt.
- Người `4`: xem **cùng bài 3 hai lần** ⇒ **không đạt**.

**Output bắt buộc:**

```text
id
--
5
6
```

## 4. Tạo database trong bộ nhớ bằng `sqlite3`

- `sqlite3.connect(':memory:')`: database tạm nằm trong RAM, không cần cài MySQL.
- `conn.execute('...')`: thực thi câu SQL; `.fetchall()` lấy các hàng kết quả dạng list các tuple.
- SQLite hỗ trợ đầy đủ `SELECT`, `GROUP BY`, `COUNT(DISTINCT)`, `HAVING`, `ORDER BY` cần cho bài này.

**Lưu ý:** SQLite là môi trường **học và kiểm thử**. Câu SQL nộp LeetCode phải chạy đúng ngữ nghĩa của MySQL; phần chính ở cuối không phụ thuộc cú pháp riêng SQLite.

In [1]:
import sqlite3
conn=sqlite3.connect(':memory:')
conn.execute('''
CREATE TABLE Views(
    article_id INTEGER,
    author_id INTEGER,
    viewer_id INTEGER,
    view_date TEXT
)
''')
print('Đã tạo bảng Views trong SQLite.')

Đã tạo bảng Views trong SQLite.


## 5. Nạp dữ liệu mẫu gốc vào bảng

`executemany` chèn nhiều hàng. Mỗi tuple tương ứng lần lượt với `article_id`, `author_id`, `viewer_id`, `view_date`.

Trong dữ liệu có **hai hàng giống nhau** `(3,4,4,'2019-07-21')` — giữ nguyên để kiểm tra trường hợp trùng lặp.

In [2]:
VIEWS_SAMPLE=[
    (1,3,5,'2019-08-01'),
    (3,4,5,'2019-08-01'),
    (1,3,6,'2019-08-02'),
    (2,7,7,'2019-08-01'),
    (2,7,6,'2019-08-02'),
    (4,7,1,'2019-07-22'),
    (3,4,4,'2019-07-21'),
    (3,4,4,'2019-07-21'),
]
conn.executemany('INSERT INTO Views VALUES (?,?,?,?)', VIEWS_SAMPLE)
rows=conn.execute('SELECT * FROM Views').fetchall()
for row in rows:
    print(row)
assert len(rows)==8

(1, 3, 5, '2019-08-01')
(3, 4, 5, '2019-08-01')
(1, 3, 6, '2019-08-02')
(2, 7, 7, '2019-08-01')
(2, 7, 6, '2019-08-02')
(4, 7, 1, '2019-07-22')
(3, 4, 4, '2019-07-21')
(3, 4, 4, '2019-07-21')


## 6. Nền tảng SQL: `SELECT`, `FROM`, `WHERE`

```sql
SELECT article_id, viewer_id
FROM Views
WHERE viewer_id = 5;
```

Đọc: lấy **hai cột** `article_id, viewer_id` từ bảng `Views`, chỉ giữ những hàng có `viewer_id=5`.

`WHERE` quyết định **giữ/bỏ từng hàng** trước khi gom nhóm, chưa trả lời được người nào xem nhiều bài.

In [3]:
q='SELECT article_id, viewer_id, view_date FROM Views WHERE viewer_id = 5;'
result=conn.execute(q).fetchall()
print('Người 5:',result)
assert result==[(1,5,'2019-08-01'),(3,5,'2019-08-01')]

Người 5: [(1, 5, '2019-08-01'), (3, 5, '2019-08-01')]


## 7. Nền tảng SQL: `ORDER BY` và đặt tên `AS`

`ORDER BY viewer_id ASC` sắp xếp từ bé tới lớn; `ASC` có thể bỏ qua vì là mặc định. `AS id` đặt tên cột kết quả theo yêu cầu LeetCode.

**Chú ý:** `ORDER BY` chỉ đổi **thứ tự kết quả**, không loại trùng hay đếm bài.

In [4]:
q='SELECT viewer_id AS id FROM Views ORDER BY id ASC;'
rows=conn.execute(q).fetchall()
print('Danh sách có thể trùng:',rows)
assert [x[0] for x in rows]==[1,4,4,5,5,6,6,7]

Danh sách có thể trùng: [(1,), (4,), (4,), (5,), (5,), (6,), (6,), (7,)]


## 8. Cách trực giác: Python gom nhóm `(viewer_id, view_date)`

Không bắt đầu bằng một câu SQL dài. Trước hết, ta tự viết ý tưởng:

1. Duyệt mọi hàng.
2. Với mỗi **cặp** `(người,ngày)`, giữ tập **article_id khác nhau**.
3. Nếu tập có từ `2` phần tử, thêm người đó vào tập kết quả.
4. Sắp xếp các id.

`dict` lưu `key -> value`; `set` tự động loại phần tử trùng. `set.add(x)` thêm `x` nếu chưa có.

Cách Python dưới đây là **baseline để hiểu đề**, không phải lời giải SQL cần nộp.

In [5]:
def article_views_python(rows):
    grouped={}
    for article_id,author_id,viewer_id,view_date in rows:
        key=(viewer_id,view_date)
        if key not in grouped:
            grouped[key]=set()
        grouped[key].add(article_id)
    answer={viewer_id for (viewer_id,_),articles in grouped.items() if len(articles)>1}
    return sorted(answer)

print('Python baseline:',article_views_python(VIEWS_SAMPLE))
assert article_views_python(VIEWS_SAMPLE)==[5,6]

Python baseline: [5, 6]


## 9. Vì sao không được gom nhóm chỉ theo người?

Một người xem bài `1` vào **thứ Hai** và bài `2` vào **thứ Ba** vẫn **không đạt**, vì đề yêu cầu **cùng ngày**.

Đơn vị nhóm đúng là **`(viewer_id, view_date)`**, không phải chỉ `viewer_id`.

In [6]:
rows_different_days=[(1,3,9,'2020-01-01'),(2,4,9,'2020-01-02')]
print('Hai bài, khác ngày =>',article_views_python(rows_different_days))
assert article_views_python(rows_different_days)==[]

Hai bài, khác ngày => []


## 10. `GROUP BY` — gom từng cặp người và ngày

```sql
SELECT viewer_id, view_date, COUNT(*) AS total_rows
FROM Views
GROUP BY viewer_id, view_date
ORDER BY viewer_id, view_date;
```

SQL đọc theo logic: từ các hàng trong `Views`, **gom những hàng có cùng `viewer_id` và `view_date` thành một nhóm**, rồi tính tổng cho mỗi nhóm.

`COUNT(*)` đếm **số hàng**, chưa chắc là số **bài viết khác nhau**.

In [7]:
SQL_GROUP_ROWS='''
SELECT viewer_id, view_date, COUNT(*) AS total_rows
FROM Views
GROUP BY viewer_id, view_date
ORDER BY viewer_id, view_date;
'''
for row in conn.execute(SQL_GROUP_ROWS):
    print(row)
assert conn.execute('SELECT COUNT(*) FROM Views WHERE viewer_id=4').fetchone()[0]==2

(1, '2019-07-22', 1)
(4, '2019-07-21', 2)
(5, '2019-08-01', 2)
(6, '2019-08-02', 2)
(7, '2019-08-01', 1)


## 11. `COUNT(*)` và `COUNT(DISTINCT article_id)` khác nhau như thế nào?

Giả sử người `4` xem **cùng bài 3 hai lần** trong ngày `2019-07-21`:

- `COUNT(*) = 2` vì có 2 hàng.
- `COUNT(DISTINCT article_id) = 1` vì chỉ có **1 mã bài khác nhau**.

`DISTINCT` loại những **giá trị trùng** trong phép đếm; đây là phần không thể thiếu của bài II.

In [8]:
SQL_COUNTS='''
SELECT viewer_id, view_date,
       COUNT(*) AS total_rows,
       COUNT(DISTINCT article_id) AS unique_articles
FROM Views
GROUP BY viewer_id, view_date
ORDER BY viewer_id, view_date;
'''
stats=conn.execute(SQL_COUNTS).fetchall()
for row in stats:
    print(row)
assert next(r for r in stats if r[0]==4)[2:]==(2,1)
assert next(r for r in stats if r[0]==5)[2:]==(2,2)

(1, '2019-07-22', 1, 1)
(4, '2019-07-21', 2, 1)
(5, '2019-08-01', 2, 2)
(6, '2019-08-02', 2, 2)
(7, '2019-08-01', 1, 1)


## 12. `HAVING` khác gì `WHERE`?

- `WHERE viewer_id=5` lọc **hàng** trước khi nhóm.
- `HAVING COUNT(DISTINCT article_id) > 1` lọc **nhóm đã tính xong số bài khác nhau**.

Chú ý điều kiện `>1` có nghĩa **ít nhất 2** bài khác nhau, không phải `>=1`.

Không thể dùng `WHERE COUNT(DISTINCT article_id)>1` như điều kiện nhóm thông thường.

In [9]:
SQL_QUALIFIED_GROUPS='''
SELECT viewer_id, view_date, COUNT(DISTINCT article_id) AS unique_articles
FROM Views
GROUP BY viewer_id, view_date
HAVING COUNT(DISTINCT article_id) > 1
ORDER BY viewer_id, view_date;
'''
qualified=conn.execute(SQL_QUALIFIED_GROUPS).fetchall()
print('Nhóm người-ngày đủ điều kiện:',qualified)
assert qualified==[(5,'2019-08-01',2),(6,'2019-08-02',2)]

Nhóm người-ngày đủ điều kiện: [(5, '2019-08-01', 2), (6, '2019-08-02', 2)]


## 13. Tại sao cần `SELECT DISTINCT viewer_id AS id` ở ngoài?

Một người có thể đủ điều kiện **vào nhiều ngày**. `GROUP BY viewer_id, view_date` lúc đó tạo nhiều nhóm cùng một người.

- `COUNT(DISTINCT article_id)` loại lặp bài **bên trong mỗi nhóm**.
- `SELECT DISTINCT viewer_id AS id` loại lặp người **giữa các nhóm đủ điều kiện**.

Đây là **hai nhiệm vụ khác nhau của `DISTINCT`**. Không nên bỏ cái nào.

In [10]:
extra=[
    (8,2,5,'2019-08-09'),
    (9,2,5,'2019-08-09'),
]
conn.executemany('INSERT INTO Views VALUES (?,?,?,?)',extra)
two_days=conn.execute(SQL_QUALIFIED_GROUPS).fetchall()
print('Người 5 xuất hiện hai ngày:',two_days)
assert sum(r[0]==5 for r in two_days)==2
# Khôi phục dữ liệu chuẩn cho các phần sau.
conn.execute('DELETE FROM Views WHERE view_date = ?',('2019-08-09',))
assert conn.execute('SELECT COUNT(*) FROM Views').fetchone()[0]==8

Người 5 xuất hiện hai ngày: [(5, '2019-08-01', 2), (5, '2019-08-09', 2), (6, '2019-08-02', 2)]


## 14. Ghép lời giải chính: chọn người, loại trùng, sắp xếp

```sql
SELECT DISTINCT viewer_id AS id
FROM Views
GROUP BY viewer_id, view_date
HAVING COUNT(DISTINCT article_id) > 1
ORDER BY id;
```

Đọc từ dưới lên theo **ý nghĩa**:

1. Nhóm từng `(viewer_id,view_date)`.
2. Giữ các nhóm có **hơn 1 article_id khác nhau**.
3. Chỉ lấy `viewer_id`, đặt tên `id` và loại **người trùng**.
4. Sắp id tăng.

In [11]:
SQL_SUBMIT='''
SELECT DISTINCT viewer_id AS id
FROM Views
GROUP BY viewer_id, view_date
HAVING COUNT(DISTINCT article_id) > 1
ORDER BY id;
'''
answer=conn.execute(SQL_SUBMIT).fetchall()
print('SQL trả:',answer)
assert answer==[(5,),(6,)]
print('Ví dụ chuẩn LeetCode: PASS')

SQL trả: [(5,), (6,)]
Ví dụ chuẩn LeetCode: PASS


## 15. Mô phỏng từng giai đoạn của câu SQL

| `(viewer_id, view_date)` | Hàng gốc | Article ID khác nhau | Sau HAVING? |
|---|---:|---|---|
| `(1, 2019-07-22)` | 1 | `{4}` | Không |
| `(4, 2019-07-21)` | **2** | `{3}` | **Không** (trùng bài) |
| `(5, 2019-08-01)` | 2 | `{1,3}` | **Có** |
| `(6, 2019-08-02)` | 2 | `{1,2}` | **Có** |
| `(7, 2019-08-01)` | 1 | `{2}` | Không |

Sau `SELECT DISTINCT ... ORDER BY id`: `5,6`.

**Thói quen tốt:** Trước khi viết SQL cuối, tự viết bảng nhóm như trên để nhìn thấy lý do từng người bị loại.

In [12]:
def trace_groups(rows):
    grouped={}
    for article_id,author_id,viewer_id,view_date in rows:
        key=(viewer_id,view_date)
        grouped.setdefault(key,[]).append(article_id)
    for key in sorted(grouped):
        raw=grouped[key]
        unique=sorted(set(raw))
        print(f'{key}: raw={raw} -> DISTINCT={unique} -> pass={len(unique)>1}')

trace_groups(VIEWS_SAMPLE)

(1, '2019-07-22'): raw=[4] -> DISTINCT=[4] -> pass=False
(4, '2019-07-21'): raw=[3, 3] -> DISTINCT=[3] -> pass=False
(5, '2019-08-01'): raw=[1, 3] -> DISTINCT=[1, 3] -> pass=True
(6, '2019-08-02'): raw=[1, 2] -> DISTINCT=[1, 2] -> pass=True
(7, '2019-08-01'): raw=[2] -> DISTINCT=[2] -> pass=False


## 16. Cách khác: dùng bảng trung gian / CTE

Với người mới, chia thành hai bước đôi khi dễ hiểu hơn:

```sql
WITH qualifying_days AS (
    SELECT viewer_id
    FROM Views
    GROUP BY viewer_id, view_date
    HAVING COUNT(DISTINCT article_id) >= 2
)
SELECT DISTINCT viewer_id AS id
FROM qualifying_days
ORDER BY id;
```

`WITH ... AS (...)` đặt tên cho **kết quả truy vấn tạm**. Cách này đúng tương đương, nhưng dài hơn lời giải chính.

In [13]:
SQL_CTE='''
WITH qualifying_days AS (
    SELECT viewer_id
    FROM Views
    GROUP BY viewer_id, view_date
    HAVING COUNT(DISTINCT article_id) >= 2
)
SELECT DISTINCT viewer_id AS id
FROM qualifying_days
ORDER BY id;
'''
assert conn.execute(SQL_CTE).fetchall()==conn.execute(SQL_SUBMIT).fetchall()
print('CTE và SQL gọn cho cùng kết quả: PASS')

CTE và SQL gọn cho cùng kết quả: PASS


## 17. Code để nộp LeetCode (MySQL)

**Bài Database không viết `class Solution`**; chọn mục **MySQL** trên LeetCode và dán **duy nhất câu SQL này**:

```sql
SELECT DISTINCT viewer_id AS id
FROM Views
GROUP BY viewer_id, view_date
HAVING COUNT(DISTINCT article_id) > 1
ORDER BY id;
```

Không dán phần Python `sqlite3` lên trang nộp SQL. Truy vấn không cần `JOIN` vì toàn bộ thông tin nằm trong cùng bảng `Views`.

In [14]:
# In câu truy vấn thực tế để copy sang trình nộp MySQL.
print(SQL_SUBMIT.strip())

SELECT DISTINCT viewer_id AS id
FROM Views
GROUP BY viewer_id, view_date
HAVING COUNT(DISTINCT article_id) > 1
ORDER BY id;


## 18. Kiểm thử các ca biên với bảng mới, không ảnh hưởng ví dụ gốc

Các ca phải nghĩ tới:

1. Không có hàng nào ⇒ không có người đạt.
2. Một người xem một bài **nhiều lần cùng ngày** ⇒ không đạt.
3. Một người xem nhiều bài nhưng ở **những ngày khác nhau** ⇒ không đạt.
4. Một người xem hai bài trong một ngày ⇒ đạt.
5. Một người đạt nhiều ngày ⇒ kết quả chỉ xuất hiện **một lần**.
6. **Tự xem bài** cũng được tính, không phải bài 1148 Article Views I.

In [15]:
def run_query_on_rows(data,sql=SQL_SUBMIT):
    test_db=sqlite3.connect(':memory:')
    try:
        test_db.execute('CREATE TABLE Views(article_id INTEGER,author_id INTEGER,viewer_id INTEGER,view_date TEXT)')
        test_db.executemany('INSERT INTO Views VALUES (?,?,?,?)',data)
        return [row[0] for row in test_db.execute(sql).fetchall()]
    finally:
        test_db.close()

cases=[
    ([],[]),
    ([(1,1,9,'2020-01-01')]*3,[]),
    ([(1,2,9,'2020-01-01'),(2,3,9,'2020-01-02')],[]),
    ([(1,2,9,'2020-01-01'),(2,3,9,'2020-01-01')],[9]),
    ([(1,2,9,'2020-01-01'),(2,3,9,'2020-01-01'),(3,4,9,'2020-01-02'),(4,5,9,'2020-01-02')],[9]),
    ([(1,9,9,'2020-01-01'),(2,9,9,'2020-01-01')],[9]),
]
for data,expected in cases:
    got=run_query_on_rows(data)
    assert got==expected,(got,expected)
print('6 ca biên: PASS')

6 ca biên: PASS


## 19. Đối chiếu ngẫu nhiên: Python baseline, SQL ngắn và SQL CTE

Tạo hàng giả ngẫu nhiên, có thể lặp. Đối chiếu ba cách viết **độc lập cú pháp** trên nhiều dữ liệu khác nhau.

In [16]:
import random
rng=random.Random(1149)
for rep in range(260):
    records=[]
    for _ in range(rng.randint(0,65)):
        records.append((rng.randint(1,8),rng.randint(1,6),rng.randint(1,12),f'2022-03-{rng.randint(1,6):02d}'))
    if records and rng.random()<0.55:
        records.extend(rng.choices(records,k=rng.randint(1,8)))
    expected=article_views_python(records)
    actual=run_query_on_rows(records)
    alternative=run_query_on_rows(records,SQL_CTE)
    assert actual==alternative==expected,(records,actual,expected)
print('260 bộ dữ liệu ngẫu nhiên: Python = SQL = CTE (PASS)')

260 bộ dữ liệu ngẫu nhiên: Python = SQL = CTE (PASS)


## 20. Giải thích đúng đắn và Big-O

**Tính đúng:** Mỗi nhóm `(viewer_id,view_date)` chứa đúng tất cả các lượt xem của một người trong một ngày. `COUNT(DISTINCT article_id)>1` tương đương người đó xem ít nhất 2 bài **khác nhau** ngày ấy. `SELECT DISTINCT viewer_id` lấy mỗi người đủ điều kiện đúng một lần. `ORDER BY id` đảm bảo tăng dần. Vì vậy kết quả **không thiếu** và **không thừa** người.

**Độ phức tạp (ước lượng thuật toán, không cam kết plan của DBMS):** đặt `N` là số hàng. Nếu hệ quản trị sắp xếp để gom nhóm, chi phí thường có thể `O(N log N)` thời gian, `O(N)` bộ nhớ trung gian; một engine có hash aggregation có thể làm gần `O(N)` trung bình cho phần gom nhóm, nhưng `ORDER BY` vẫn có chi phí sắp xếp kết quả. Thực tế phụ thuộc chỉ mục/optimizer.

**Không nói SQL chắc chắn `O(N)`** vì không biết kế hoạch thực thi của MySQL.

## 21. Các lỗi dễ mắc

| SQL sai | Vì sao sai | Sửa |
|---|---|---|
| `GROUP BY viewer_id` | Gom tất cả ngày lại | Gom theo `viewer_id, view_date` |
| `HAVING COUNT(*)>1` | Xem 1 bài nhiều lần cũng qua | `COUNT(DISTINCT article_id)>1` |
| `WHERE author_id=viewer_id` | Đó là bài khác (#1148) | Bỏ điều kiện tự xem |
| `SELECT viewer_id` thiếu `DISTINCT` | Người đạt nhiều ngày có thể xuất hiện nhiều lần | `SELECT DISTINCT viewer_id AS id` |
| Quên `ORDER BY` | Không bảo đảm thứ tự output | `ORDER BY id` |
| Dùng `WHERE COUNT(...)` | `WHERE` không lọc kết quả aggregate | Dùng `HAVING` |

In [17]:
wrong_count='''
SELECT DISTINCT viewer_id AS id
FROM Views
GROUP BY viewer_id, view_date
HAVING COUNT(*) > 1
ORDER BY id;
'''
correct=[r[0] for r in conn.execute(SQL_SUBMIT)]
wrong=[r[0] for r in conn.execute(wrong_count)]
print('Đúng:',correct,'| Sai do COUNT(*) (thừa người 4):',wrong)
assert correct==[5,6] and wrong==[4,5,6]

Đúng: [5, 6] | Sai do COUNT(*) (thừa người 4): [4, 5, 6]


## 22. Mẹo nhận dạng pattern

Đề có các dấu hiệu:

- **"cùng ngày"** ⇒ ngày phải nằm trong khóa nhóm.
- **"nhiều đối tượng khác nhau"** ⇒ `COUNT(DISTINCT object_id)`.
- **"ít nhất K" / "hơn K"** ⇒ `HAVING COUNT(...) >= K` / `> K`.
- **"mỗi người chỉ một dòng"** ⇒ `SELECT DISTINCT person_id` hoặc nhóm thêm ở ngoài.
- **"sắp tăng"** ⇒ `ORDER BY ... ASC`.

Áp dụng tương tự cho đăng nhập nhiều thiết bị trong một ngày, mua nhiều loại hàng, xem nhiều phim, v.v.

In [18]:
# Ví dụ biến thể: người xem ít nhất 3 bài KHÁC NHAU trong một ngày.
sql_three=SQL_SUBMIT.replace('> 1','>= 3')
data_three=[(1,1,8,'2022-02-02'),(2,1,8,'2022-02-02'),(3,1,8,'2022-02-02')]
print('Viewer xem >=3 bài khác nhau:',run_query_on_rows(data_three,sql_three))
assert run_query_on_rows(data_three,sql_three)==[8]

Viewer xem >=3 bài khác nhau: [8]


## 23. Tự luyện — 5 bài SQL tăng dần

**Bài 1 — Đoán kết quả:** Một người xem bài 10 ba lần cùng ngày. Kết quả có người này không? `COUNT(*)` và `COUNT(DISTINCT article_id)` lần lượt là bao nhiêu?

**Bài 2 — Tính bằng tay:** Người 8 xem bài 1 ngày 1, bài 2 ngày 2; người 9 xem bài 1 và 3 ngày 2. Ai được trả về?

**Bài 3 — TODO:** Viết SQL liệt kê các cặp `(viewer_id,view_date)` kèm `number_of_articles` đếm bài khác nhau, sắp theo người và ngày.

**Bài 4 — TODO:** Viết truy vấn trả về người xem **ít nhất 3 bài khác nhau trong cùng ngày**.

**Bài 5 — Biến thể:** Viết truy vấn trả về `id` của những người có ít nhất **2 ngày khác nhau**, mỗi ngày họ xem ít nhất 2 bài khác nhau. Dùng CTE tạo danh sách ngày đạt rồi nhóm người lần nữa.

Hãy tự viết SQL trong các chuỗi TODO, chạy bằng helper ở trên; phần đáp án ở cuối.

In [19]:
# TODO: chuỗi trống/có chú thích không được tự thực thi.
EX_1149_3='''-- TODO: SELECT viewer_id, view_date, COUNT(DISTINCT ...) ... GROUP BY ...'''
EX_1149_4='''-- TODO: HAVING COUNT(DISTINCT article_id) >= 3'''
EX_1149_5='''-- TODO: WITH days AS (...) SELECT ... GROUP BY ... HAVING COUNT(*) >= 2'''

# Khi làm xong Bài 3:
# print(run_query_on_rows(VIEWS_SAMPLE, EX_1149_3))  # Sửa helper nếu cần nhiều cột.

## 24. Gợi ý 3 mức

| Bài | Mức 1 — nghĩ gì | Mức 2 — SQL nào | Mức 3 — gần đáp án |
|---|---|---|---|
| 1 | Cùng bài != nhiều bài | `COUNT(*)`, `COUNT(DISTINCT ...)` | `3` và `1`, không đạt |
| 2 | Ghép người–ngày | `GROUP BY viewer_id,view_date` | Chỉ người `9` |
| 3 | Đếm mọi nhóm kể cả 1 bài | `GROUP BY` + `COUNT(DISTINCT)` | Không thêm `HAVING` |
| 4 | Đếm số bài tối thiểu | `HAVING COUNT(DISTINCT article_id)` | `>= 3` |
| 5 | Hai cấp nhóm | CTE cho ngày, `GROUP BY viewer_id` ở ngoài | `HAVING COUNT(*) >= 2` trên các ngày đạt |

## 25. Đáp án tham khảo — chỉ mở sau khi tự làm

**Bài 1:** `COUNT(*)=3`, `COUNT(DISTINCT article_id)=1` ⇒ người này không đạt.

**Bài 2:** Chỉ người `9`, vì người `8` xem khác ngày.

**Bài 3–5:** các chuỗi SQL trong cell sau. Bài 5 sẽ được kiểm tra bằng dữ liệu riêng có người đạt hai ngày.

In [20]:
ANSWER_1149_3='''
SELECT viewer_id, view_date,
       COUNT(DISTINCT article_id) AS number_of_articles
FROM Views
GROUP BY viewer_id, view_date
ORDER BY viewer_id, view_date;
'''
ANSWER_1149_4='''
SELECT DISTINCT viewer_id AS id
FROM Views
GROUP BY viewer_id, view_date
HAVING COUNT(DISTINCT article_id) >= 3
ORDER BY id;
'''
ANSWER_1149_5='''
WITH qualifying_days AS (
    SELECT viewer_id, view_date
    FROM Views
    GROUP BY viewer_id, view_date
    HAVING COUNT(DISTINCT article_id) >= 2
)
SELECT viewer_id AS id
FROM qualifying_days
GROUP BY viewer_id
HAVING COUNT(*) >= 2
ORDER BY id;
'''
assert len(conn.execute(ANSWER_1149_3).fetchall())==5
assert conn.execute(ANSWER_1149_4).fetchall()==[]
two_day_records=[
    (1,1,9,'2022-01-01'),(2,1,9,'2022-01-01'),
    (3,1,9,'2022-01-02'),(4,1,9,'2022-01-02'),
    (1,1,8,'2022-01-01'),(2,1,8,'2022-01-01'),
]
assert run_query_on_rows(two_day_records,ANSWER_1149_5)==[9]
print('Đáp án 3–5: PASS')

Đáp án 3–5: PASS


## 26. Tổng kết và checklist submit

```sql
SELECT DISTINCT viewer_id AS id
FROM Views
GROUP BY viewer_id, view_date
HAVING COUNT(DISTINCT article_id) > 1
ORDER BY id;
```

- [ ] Đúng bảng `Views`, không cần `JOIN`.
- [ ] Nhóm theo **cả người và ngày**.
- [ ] `COUNT(DISTINCT article_id)` đếm **bài khác nhau**, không phải hàng.
- [ ] `HAVING ... > 1` vì đề yêu cầu hơn một bài.
- [ ] `SELECT DISTINCT viewer_id AS id` để không trùng người nhiều ngày.
- [ ] `ORDER BY id` tăng dần.
- [ ] Dán **SQL** vào trình nộp **MySQL**; không dán Python SQLite demo.
- [ ] Đã thử bảng trùng hàng, khác ngày, và người tự xem bài.